In [30]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
BASE_PATH = r"D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tareaunidad4"

COMBINED_FILES = [os.path.join(BASE_PATH, f"combined_data_{i}.txt") for i in range(1, 5)]
MOVIE_TITLES = os.path.join(BASE_PATH, "movie_titles.csv")
OUT_DIR = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

In [40]:
def iter_combined(path: str):
    """
    Itera sobre un archivo combined_data_X.txt y devuelve (user, movie, rating)
    Maneja líneas con errores o formatos irregulares.
    """
    movie_id = None
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            # Si la línea termina con ":", es un nuevo MovieID
            if s.endswith(":"):
                try:
                    movie_id = int(s[:-1])
                except ValueError:
                    continue
            else:
                parts = s.split(",")
                # Validar que tenga exactamente 3 columnas
                if len(parts) != 3:
                    continue
                try:
                    user = int(parts[0])
                    rating = int(parts[1])
                    # parts[2] es la fecha, pero no la usamos
                    yield user, movie_id, rating
                except ValueError:
                    # Si algo no se puede convertir a número, se salta
                    continue


def load_sample(files, n_users=4000, seed=42):
    """Carga subset de usuarios para reducir tamaño"""
    np.random.seed(seed)
    users_seen = set()
    for path in files:
        for u, m, r in iter_combined(path):
            users_seen.add(u)
    sample_users = set(np.random.choice(list(users_seen), n_users, replace=False))
    data = []
    for path in files:
        for u, m, r in iter_combined(path):
            if u in sample_users:
                data.append((u, m, r))
    df = pd.DataFrame(data, columns=["UserID", "MovieID", "Rating"])
    print(f"Cargados {len(df)} ratings | {df['UserID'].nunique()} usuarios | {df['MovieID'].nunique()} películas")
    return df

In [33]:
def build_matrix(df: pd.DataFrame):
    """Crea matriz usuario-película dispersa"""
    user_map = {u: i for i, u in enumerate(sorted(df["UserID"].unique()))}
    movie_map = {m: i for i, m in enumerate(sorted(df["MovieID"].unique()))}
    rows = df["UserID"].map(user_map)
    cols = df["MovieID"].map(movie_map)
    vals = df["Rating"].astype(float)
    R = csr_matrix((vals, (rows, cols)), shape=(len(user_map), len(movie_map)))
    return R, user_map, movie_map

def top_k_similar(sim_vector, k=5):
    """Devuelve índices y pesos de los k más similares (ya se recibe un vector 1D)."""
    sim_vector = sim_vector.copy()
    sim_vector[np.isnan(sim_vector)] = 0
    top_idx = np.argsort(sim_vector)[::-1][:k]
    top_vals = sim_vector[top_idx]
    return top_idx, top_vals

In [34]:
def recommend_user_based(R, user_map, movie_map, user_id, k=10, n_recs=10):
    """
    Filtrado colaborativo basado en usuarios.
    Calcula similitud coseno entre el usuario y los demás.
    """
    R_dense = R.toarray()
    user_index = user_map[user_id]

    # Calculamos similitud coseno entre el usuario y todos los demás usuarios
    user_sim = cosine_similarity(R_dense, R_dense[user_index].reshape(1, -1)).ravel()
    user_sim[user_index] = 0  # el mismo usuario no cuenta

    # Top-k usuarios similares
    top_users, weights = top_k_similar(user_sim, k=k)

    # Promedio ponderado de calificaciones
    numerator = np.dot(weights, R_dense[top_users])
    denominator = np.abs(weights).sum() + 1e-9
    user_ratings = numerator / denominator

    # Excluir películas ya vistas
    seen = set(np.where(R_dense[user_index] > 0)[0])
    candidates = [(i, score) for i, score in enumerate(user_ratings) if i not in seen]

    # Top-N recomendaciones
    top = sorted(candidates, key=lambda x: x[1], reverse=True)[:n_recs]
    inv_movie_map = {v: k for k, v in movie_map.items()}
    return [(inv_movie_map[i], float(score)) for i, score in top]

def recommend_item_based(R, user_map, movie_map, user_id, k=10, n_recs=10):
    """Filtrado colaborativo basado en ítems con normalización (escala ~1-5)."""
    R_dense = R.toarray()
    user_index = user_map[user_id]
    user_ratings = R_dense[user_index]  # vector de ratings del usuario (size = M items)

    # Similitud item-item (M x M)
    item_sim = cosine_similarity(R_dense.T)  # cuidado con memoria si M grande

    # Para cada item i, calcular: sum_j sim(i,j)*r_uj  /  sum_j |sim(i,j)|   sobre j que el usuario ha visto
    seen_idx = np.where(user_ratings > 0)[0]
    if seen_idx.size == 0:
        return []  # sin datos

    # Numerador y denominador vectorizados
    numer = item_sim[:, seen_idx] @ user_ratings[seen_idx]
    denom = np.abs(item_sim[:, seen_idx]).sum(axis=1) + 1e-9
    scores = numer / denom  # ahora está en una escala comparable

    # Excluir items ya vistos
    seen = set(seen_idx.tolist())
    candidates = [(i, scores[i]) for i in range(len(scores)) if i not in seen]

    # Top-N
    top = sorted(candidates, key=lambda x: x[1], reverse=True)[:n_recs]
    inv_movie_map = {v: k for k, v in movie_map.items()}
    return [(inv_movie_map[i], float(np.clip(s, 1.0, 5.0))) for i, s in top]

In [35]:
def main():
    print("=== Cargando datos ===")
    df = load_sample(COMBINED_FILES, n_users=3000)

    print("=== Construyendo matriz ===")
    R, user_map, movie_map = build_matrix(df)
    inv_user_map = {v: k for k, v in user_map.items()}

    # Usuario de ejemplo (el que más calificaciones hizo)
    example_user = df["UserID"].value_counts().idxmax()
    print(f"Ejemplo de usuario: {example_user}")

    print("=== Recomendaciones USER-BASED ===")
    user_recs = recommend_user_based(R, user_map, movie_map, example_user, k=10, n_recs=10)
    user_df = pd.DataFrame(user_recs, columns=["MovieID", "PredScore"])

    print("=== Recomendaciones ITEM-BASED ===")
    item_recs = recommend_item_based(R, user_map, movie_map, example_user, k=10, n_recs=10)
    item_df = pd.DataFrame(item_recs, columns=["MovieID", "PredScore"])

    if os.path.exists(MOVIE_TITLES):
        titles = pd.read_csv(
    MOVIE_TITLES,
    header=None,
    names=["MovieID", "Year", "Title"],
    encoding="ISO-8859-1",
    engine="python",           # usa parser más tolerante
    on_bad_lines="skip",       # salta líneas con errores
    quotechar='"',             # maneja comillas
    sep=",",                   # separador explícito 
    )
        user_df = user_df.merge(titles, on="MovieID", how="left")
        item_df = item_df.merge(titles, on="MovieID", how="left")

    user_out = os.path.join(OUT_DIR, f"user_based_{example_user}.csv")
    item_out = os.path.join(OUT_DIR, f"item_based_{example_user}.csv")
    user_df.to_csv(user_out, index=False)
    item_df.to_csv(item_out, index=False)

    print(f"\nTop-10 (user-based) guardado en: {user_out}")
    print(f"Top-10 (item-based) guardado en: {item_out}")
    print("Proceso completado ")

if __name__ == "__main__":
    main()

=== Cargando datos ===
Cargados 622942 ratings | 3000 usuarios | 15096 películas
=== Construyendo matriz ===
Ejemplo de usuario: 16272
=== Recomendaciones USER-BASED ===
=== Recomendaciones ITEM-BASED ===

Top-10 (user-based) guardado en: D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tareaunidad4\outputs\user_based_16272.csv
Top-10 (item-based) guardado en: D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tareaunidad4\outputs\item_based_16272.csv
Proceso completado ✅


In [42]:
from math import sqrt

def compute_rmse(R, user_map, movie_map, df_test, model_func, **kwargs):
    """
    Calcula RMSE usando el método indicado (user-based o item-based).
    df_test: DataFrame con columnas UserID, MovieID, Rating.
    model_func: función predictora (recommend_user_based o recommend_item_based)
    kwargs: parámetros (k, n_recs, etc.)
    """
    errors = []
    count = 0
    for _, row in df_test.iterrows():
        u, m, real = row["UserID"], row["MovieID"], row["Rating"]
        if u in user_map and m in movie_map:
            pred_list, *_ = model_func(R, user_map, movie_map, u, **kwargs)
            if isinstance(pred_list, list):
                pred_dict = dict(pred_list)
                if m in pred_dict:
                    pred = pred_dict[m]
                else:
                    # Si no está en las recomendaciones, saltamos
                    continue
                errors.append((real - pred) ** 2)
                count += 1
    return sqrt(sum(errors) / len(errors)) if errors else None


In [43]:
print("=== CARGANDO DATOS ===")
df = load_sample(COMBINED_FILES, n_users=3000)

# Dividimos train/test
mask = np.random.rand(len(df)) < 0.8
train_df = df[mask]
test_df = df[~mask]
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

# Matriz con datos de entrenamiento
R, user_map, movie_map = build_matrix(train_df)
inv_user_map = {v: k for k, v in user_map.items()}

example_user = train_df["UserID"].value_counts().idxmax()
print(f"Usuario ejemplo: {example_user}")


=== CARGANDO DATOS ===
Cargados 622942 ratings | 3000 usuarios | 15096 películas
Train: 498522, Test: 124420
Usuario ejemplo: 16272


In [44]:
print("\n=== EVALUANDO RMSE ===")
rmse_user = compute_rmse(R, user_map, movie_map, test_df, recommend_user_based, k=10, n_recs=10)
rmse_item = compute_rmse(R, user_map, movie_map, test_df, recommend_item_based, k=10, n_recs=10)
print(f"RMSE User-based: {rmse_user:.4f}")
print(f"RMSE Item-based: {rmse_item:.4f}")



=== EVALUANDO RMSE ===


KeyboardInterrupt: 

Usuario 

In [37]:
def iter_combined(path: str):
    """Itera sobre un archivo combined_data_X.txt de Netflix Prize."""
    movie_id = None
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s.endswith(":"):
                # Es el ID de la película (línea con ':')
                movie_id = int(s[:-1])
            else:
                parts = s.split(",")
                if len(parts) != 3:
                    # Línea inválida, la saltamos
                    continue
                user, rating, date = parts
                yield int(user), int(movie_id), int(rating), date

#  Buscar todas las películas vistas por un usuario 
def get_user_ratings(user_id, files):
    data = []
    for path in files:
        for u, m, r, d in iter_combined(path):
            if u == user_id:
                data.append((m, r, d))
    df = pd.DataFrame(data, columns=["MovieID", "Rating", "Date"])
    return df

user_id = 16272
user_movies = get_user_ratings(user_id, COMBINED_FILES)
print(f"El usuario {user_id} calificó {len(user_movies)} películas.")
print(user_movies.head())


titles = pd.read_csv(
    MOVIE_TITLES,
    header=None,
    names=["MovieID", "Year", "Title"],
    encoding="ISO-8859-1",
    engine="python",
    sep=",",
    quotechar='"',
    on_bad_lines="skip"
)

user_movies = user_movies.merge(titles, on="MovieID", how="left")
print(user_movies.head(10))


El usuario 16272 calificó 5900 películas.
   MovieID  Rating        Date
0        1       4  2005-01-20
1        3       4  2003-02-13
2        4       2  2003-12-12
3       16       3  2002-08-20
4       17       3  2005-09-07
   MovieID  Rating        Date    Year                         Title
0        1       4  2005-01-20  2003.0               Dinosaur Planet
1        3       4  2003-02-13  1997.0                     Character
2        4       2  2003-12-12  1994.0  Paula Abdul's Get Up & Dance
3       16       3  2002-08-20  1996.0                     Screamers
4       17       3  2005-09-07  2005.0                     7 Seconds
5       18       4  2002-12-06  1994.0              Immortal Beloved
6       19       3  2002-12-06  2000.0         By Dawn's Early Light
7       24       2  2002-08-27  1981.0           My Bloody Valentine
8       28       2  2003-03-14  2002.0               Lilo and Stitch
9       30       4  2005-04-25  2003.0        Something's Gotta Give
